# Improving Standalone Positioning: Open Sky vs Wall Proximity

This notebook processes two datasets:

1. `SEPT0640.26O` = open sky dataset  
2. `SEPT0641.26O` = wall proximity dataset  

It runs the same implementation chain for both:

1. Baseline `OLS_L1`  
2. `WLS_soft_L1`  
3. `OLS_IF`  
4. `HATCH_IF`  

It saves the essential plots and CSV summaries needed for the short report.

Run this notebook inside your `lab6` folder so it can import `rinexReader.py` and `SatOrbits.py`.


In [ ]:
# ============================================================
# 1. Imports and user settings
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rinexReader as rr
import SatOrbits as so

BASE_DIR = r"d:\M.Sc. Autonomous Systems - DTU\Spring Semester\30554 GNSS\Lab\lab6"

SP3_FILE = os.path.join(BASE_DIR, "COD0OPSRAP_20260640000_01D_05M_ORB.SP3")

DATASETS = {
    "open_sky": {
        "rinex": os.path.join(BASE_DIR, "SEPT0640.26O"),
        "label": "Open sky"
    },
    "wall_proximity": {
        "rinex": os.path.join(BASE_DIR, "SEPT0641.26O"),
        "label": "Wall proximity"
    }
}

OUTPUT_DIR = os.path.join(BASE_DIR, "report_outputs_open_vs_wall")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
CSV_DIR = os.path.join(OUTPUT_DIR, "csv")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

L1_CODE = "C1C"
L2_CODE = "C2W"
L1_PHASE = "L1C"
L2_PHASE = "L2W"

SIG_TYPES = [L1_CODE, L2_CODE, L1_PHASE, L2_PHASE]
CONSTS = ["G"]

CLIGHT = 299792458.0
F1 = 1575.42e6
F2 = 1227.60e6
LAMBDA_L1 = CLIGHT / F1
LAMBDA_L2 = CLIGHT / F2

X0 = np.array([0.0, 0.0, 0.0, 0.0])

SOFT_WEIGHT_FLOOR = 0.3
MIN_WEIGHT = 1e-4
HATCH_WINDOW = 30
HATCH_RESET_THRESHOLD_M = 20.0

plt.rcParams["figure.dpi"] = 120


In [ ]:
# ============================================================
# 2. Helper functions
# ============================================================

def read_approx_position_from_rinex(filepath):
    with open(filepath, "r", errors="ignore") as f:
        for line in f:
            if "APPROX POSITION XYZ" in line:
                values = line[:60].split()
                return np.array([float(values[0]), float(values[1]), float(values[2])], dtype=float)
    raise ValueError("APPROX POSITION XYZ not found in RINEX header.")


def ecef_to_geodetic(x, y, z):
    a = 6378137.0
    f = 1.0 / 298.257223563
    e2 = f * (2.0 - f)

    lon = np.arctan2(y, x)
    p = np.sqrt(x**2 + y**2)
    lat = np.arctan2(z, p * (1.0 - e2))

    for _ in range(10):
        N = a / np.sqrt(1.0 - e2 * np.sin(lat)**2)
        h = p / np.cos(lat) - N
        lat_new = np.arctan2(z, p * (1.0 - e2 * N / (N + h)))
        if abs(lat_new - lat) < 1e-12:
            lat = lat_new
            break
        lat = lat_new

    N = a / np.sqrt(1.0 - e2 * np.sin(lat)**2)
    h = p / np.cos(lat) - N
    return lat, lon, h


def ecef_to_enu_matrix(ref_xyz):
    lat, lon, _ = ecef_to_geodetic(ref_xyz[0], ref_xyz[1], ref_xyz[2])
    slat, clat = np.sin(lat), np.cos(lat)
    slon, clon = np.sin(lon), np.cos(lon)

    return np.array([
        [-slon,          clon,          0.0],
        [-slat * clon,  -slat * slon,   clat],
        [ clat * clon,   clat * slon,   slat]
    ])


def ecef_delta_to_enu(delta_xyz, ref_xyz):
    R = ecef_to_enu_matrix(ref_xyz)
    if delta_xyz.ndim == 1:
        return R @ delta_xyz
    return (R @ delta_xyz.T).T


def compute_satellite_elevations(satpos, receiver_xyz):
    sat_xyz = satpos.iloc[:, :3].to_numpy(dtype=float)
    rec_xyz = np.asarray(receiver_xyz, dtype=float)

    los_ecef = sat_xyz - rec_xyz[None, :]
    enu = ecef_delta_to_enu(los_ecef, rec_xyz)

    horizontal = np.sqrt(enu[:, 0]**2 + enu[:, 1]**2)
    elevation = np.arctan2(enu[:, 2], horizontal)
    return elevation


def weights_soft(elevation, floor=SOFT_WEIGHT_FLOOR):
    elevation = np.clip(elevation, 0.0, np.pi / 2.0)
    weights = floor + (1.0 - floor) * np.sin(elevation)
    return np.maximum(weights, MIN_WEIGHT)


def ionosphere_free_combination(v1, v2):
    f1_sq = F1 ** 2
    f2_sq = F2 ** 2
    return (f1_sq * v1 - f2_sq * v2) / (f1_sq - f2_sq)


def create_kernel(obs, satpos, x):
    x = np.asarray(x, dtype=float)

    dx = satpos.iloc[:, 0].to_numpy(dtype=float) - x[0]
    dy = satpos.iloc[:, 1].to_numpy(dtype=float) - x[1]
    dz = satpos.iloc[:, 2].to_numpy(dtype=float) - x[2]

    rng = np.sqrt(dx**2 + dy**2 + dz**2)
    los = np.column_stack((dx / rng, dy / rng, dz / rng))
    A = np.hstack((-los, np.ones((len(los), 1))))
    L = obs.iloc[:, 0].to_numpy(dtype=float) - rng - x[3]
    return L, A


def solve_ls(A, L, weights=None):
    if weights is None:
        N = A.T @ A
        u = A.T @ L
    else:
        weights = np.asarray(weights, dtype=float)
        weights = np.maximum(weights, MIN_WEIGHT)
        W = np.diag(weights)
        N = A.T @ W @ A
        u = A.T @ W @ L

    try:
        return np.linalg.solve(N, u)
    except np.linalg.LinAlgError:
        return np.linalg.lstsq(N, u, rcond=None)[0]


def spp_solver(obs, satpos, x0, weights=None):
    tol = 0.001
    maxiter = 50
    x = np.asarray(x0, dtype=float).copy()
    h = np.array([100.0, 100.0, 100.0])
    curiter = 0

    while np.sum(np.abs(h)) > tol and curiter < maxiter:
        L, A = create_kernel(obs, satpos, x)
        dx = solve_ls(A, L, weights=weights)
        x = x + dx
        h = dx[:3]
        curiter += 1

    return pd.Series(x, index=["X", "Y", "Z", "cdt"])


def hatch_update_for_satellite(sat_id, code_m, phase_m, hatch_state,
                               window=HATCH_WINDOW,
                               reset_threshold=HATCH_RESET_THRESHOLD_M):
    if sat_id not in hatch_state:
        hatch_state[sat_id] = {
            "P_smooth": code_m,
            "P_prev": code_m,
            "Phi_prev": phase_m,
            "count": 1
        }
        return code_m, 1

    state = hatch_state[sat_id]
    code_change = code_m - state["P_prev"]
    phase_change = phase_m - state["Phi_prev"]

    if abs(code_change - phase_change) > reset_threshold:
        hatch_state[sat_id] = {
            "P_smooth": code_m,
            "P_prev": code_m,
            "Phi_prev": phase_m,
            "count": 1
        }
        return code_m, 1

    count = min(state["count"] + 1, window)
    alpha = 1.0 / count

    P_smooth = alpha * code_m + (1.0 - alpha) * (state["P_smooth"] + phase_change)

    hatch_state[sat_id] = {
        "P_smooth": P_smooth,
        "P_prev": code_m,
        "Phi_prev": phase_m,
        "count": count
    }

    return P_smooth, 0


def apply_hatch_filter_epoch(obs_if, phase_if_m, hatch_state):
    smoothed_values = []
    reset_count = 0

    for sat_id in obs_if.index:
        code_m = float(obs_if.loc[sat_id, "P_IF"])
        phase_m = float(phase_if_m.loc[sat_id])
        P_smooth, reset_flag = hatch_update_for_satellite(sat_id, code_m, phase_m, hatch_state)
        smoothed_values.append(P_smooth)
        reset_count += reset_flag

    obs_hatch_if = pd.DataFrame(smoothed_values, index=obs_if.index, columns=["P_IF_HATCH"])
    return obs_hatch_if, reset_count


def solution_dict_to_dataframe(sol_dict):
    df = pd.DataFrame(sol_dict).T
    if "X" not in df.columns and "X" in df.index:
        df = df.T
    df = df[["X", "Y", "Z", "cdt"]]
    df.index = pd.to_datetime(df.index, errors="coerce")
    df = df[~df.index.isna()]
    return df.sort_index()


def compute_error_dataframe(solution_df, ref_xyz):
    xyz = solution_df[["X", "Y", "Z"]].to_numpy(dtype=float)
    delta_xyz = xyz - ref_xyz[None, :]
    enu = ecef_delta_to_enu(delta_xyz, ref_xyz)

    err = pd.DataFrame(index=solution_df.index)
    err["dX"] = delta_xyz[:, 0]
    err["dY"] = delta_xyz[:, 1]
    err["dZ"] = delta_xyz[:, 2]
    err["E"] = enu[:, 0]
    err["N"] = enu[:, 1]
    err["U"] = enu[:, 2]
    err["horizontal_error"] = np.sqrt(err["E"]**2 + err["N"]**2)
    err["vertical_error"] = np.abs(err["U"])
    err["3d_error"] = np.sqrt(err["E"]**2 + err["N"]**2 + err["U"]**2)
    return err


def rms(values):
    values = np.asarray(values, dtype=float)
    return np.sqrt(np.nanmean(values**2))


def summarize_errors(error_results, scenario_key, scenario_label):
    rows = []
    for method, err in error_results.items():
        rows.append({
            "scenario_key": scenario_key,
            "scenario": scenario_label,
            "method": method,
            "epochs": len(err),
            "mean_horizontal_error_m": np.nanmean(err["horizontal_error"]),
            "rms_horizontal_error_m": rms(err["horizontal_error"]),
            "mean_vertical_error_m": np.nanmean(err["vertical_error"]),
            "rms_vertical_error_m": rms(err["vertical_error"]),
            "rms_3d_error_m": rms(err["3d_error"]),
            "std_E_m": np.nanstd(err["E"]),
            "std_N_m": np.nanstd(err["N"]),
            "std_U_m": np.nanstd(err["U"]),
            "max_horizontal_error_m": np.nanmax(err["horizontal_error"]),
            "max_3d_error_m": np.nanmax(err["3d_error"])
        })
    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# 3. Main processing function
# ============================================================

def process_dataset(dataset_key, rinex_path, scenario_label):
    print("=" * 80)
    print(f"Processing dataset: {scenario_label}")
    print(f"RINEX: {rinex_path}")
    print("=" * 80)

    ref_xyz = read_approx_position_from_rinex(rinex_path)
    print("Reference ECEF position from RINEX header:")
    print(ref_xyz)

    rinexFile = rr.rinexReader(rinex_path)
    svpos = so.sp3Orbits(SP3_FILE)

    print(f"Reading observations: {SIG_TYPES}")
    rinexFile.readFile(CONSTS, SIG_TYPES)

    solutions = {
        "OLS_L1": {},
        "WLS_soft_L1": {},
        "OLS_IF": {},
        "HATCH_IF": {}
    }

    last_x = {
        "OLS_L1": X0.copy(),
        "WLS_soft_L1": X0.copy(),
        "OLS_IF": X0.copy(),
        "HATCH_IF": X0.copy()
    }

    hatch_state = {}
    info = {}

    start_time = time.time()
    print("Computing solutions: ", end="")

    for epoch in rinexFile.timelist:
        print(".", end="")

        obs = rinexFile.get_epoch_data(epoch, oTypes=SIG_TYPES)
        obs = obs.dropna()

        if len(obs) < 4:
            continue

        tau = obs[L1_CODE] / CLIGHT
        satpos_full = svpos.getSvPos(epoch, tau)

        if len(satpos_full) < 4:
            continue

        if len(satpos_full) != len(obs):
            continue

        cdts = satpos_full.iloc[:, 3] * CLIGHT
        satpos = satpos_full.iloc[:, :3]

        # 1. OLS L1
        obs_l1 = obs[[L1_CODE]].copy()
        obs_l1_corr = obs_l1.copy()
        obs_l1_corr.iloc[:, :] = obs_l1_corr.to_numpy(dtype=float) + cdts.values[:, None]

        x_ols_l1 = spp_solver(obs_l1_corr, satpos, last_x["OLS_L1"], weights=None)
        solutions["OLS_L1"][epoch] = x_ols_l1
        last_x["OLS_L1"] = x_ols_l1.to_numpy(dtype=float)

        # 2. WLS soft L1
        elevation = compute_satellite_elevations(
            satpos,
            x_ols_l1[["X", "Y", "Z"]].to_numpy(dtype=float)
        )
        w_soft = weights_soft(elevation)

        x_wls_l1 = spp_solver(obs_l1_corr, satpos, x_ols_l1.to_numpy(dtype=float), weights=w_soft)
        solutions["WLS_soft_L1"][epoch] = x_wls_l1
        last_x["WLS_soft_L1"] = x_wls_l1.to_numpy(dtype=float)

        # 3. OLS IF
        p1 = obs[L1_CODE].to_numpy(dtype=float)
        p2 = obs[L2_CODE].to_numpy(dtype=float)

        p1_corr = p1 + cdts.to_numpy(dtype=float)
        p2_corr = p2 + cdts.to_numpy(dtype=float)

        p_if = ionosphere_free_combination(p1_corr, p2_corr)
        obs_if = pd.DataFrame(p_if, index=obs.index, columns=["P_IF"])

        x_ols_if = spp_solver(obs_if, satpos, last_x["OLS_IF"], weights=None)
        solutions["OLS_IF"][epoch] = x_ols_if
        last_x["OLS_IF"] = x_ols_if.to_numpy(dtype=float)

        # 4. Hatch IF
        phi1_m = obs[L1_PHASE].to_numpy(dtype=float) * LAMBDA_L1
        phi2_m = obs[L2_PHASE].to_numpy(dtype=float) * LAMBDA_L2

        phi1_corr_m = phi1_m + cdts.to_numpy(dtype=float)
        phi2_corr_m = phi2_m + cdts.to_numpy(dtype=float)

        phi_if_m = ionosphere_free_combination(phi1_corr_m, phi2_corr_m)
        phase_if = pd.Series(phi_if_m, index=obs.index, name="PHI_IF")

        obs_hatch_if, reset_count = apply_hatch_filter_epoch(obs_if, phase_if, hatch_state)

        x_hatch_if = spp_solver(obs_hatch_if, satpos, last_x["HATCH_IF"], weights=None)
        solutions["HATCH_IF"][epoch] = x_hatch_if
        last_x["HATCH_IF"] = x_hatch_if.to_numpy(dtype=float)

        info[epoch] = pd.Series({
            "num_sats": len(obs),
            "mean_elev_deg": np.nanmean(np.rad2deg(elevation)),
            "min_elev_deg": np.nanmin(np.rad2deg(elevation)),
            "hatch_resets": reset_count
        })

    print("")
    print(f"Processing time: {time.time() - start_time:.3f} seconds")

    solution_results = {}
    for method, sol_dict in solutions.items():
        if len(sol_dict) > 0:
            solution_results[method] = solution_dict_to_dataframe(sol_dict)

    info_df = pd.DataFrame(info).T
    info_df.index = pd.to_datetime(info_df.index, errors="coerce")
    info_df = info_df[~info_df.index.isna()]
    info_df = info_df.sort_index()

    error_results = {}
    for method, df in solution_results.items():
        error_results[method] = compute_error_dataframe(df, ref_xyz)

    summary = summarize_errors(error_results, dataset_key, scenario_label)

    scenario_csv_dir = os.path.join(CSV_DIR, dataset_key)
    os.makedirs(scenario_csv_dir, exist_ok=True)

    summary.to_csv(os.path.join(scenario_csv_dir, f"summary_{dataset_key}.csv"), index=False)
    info_df.to_csv(os.path.join(scenario_csv_dir, f"epoch_info_{dataset_key}.csv"))

    for method, df in solution_results.items():
        df.to_csv(os.path.join(scenario_csv_dir, f"solution_{method}_{dataset_key}.csv"))

    for method, err in error_results.items():
        err.to_csv(os.path.join(scenario_csv_dir, f"errors_{method}_{dataset_key}.csv"))

    print("Summary:")
    display(summary)

    return {
        "dataset_key": dataset_key,
        "scenario_label": scenario_label,
        "ref_xyz": ref_xyz,
        "solutions": solution_results,
        "errors": error_results,
        "summary": summary,
        "info": info_df
    }


In [ ]:
# ============================================================
# 4. Run both datasets
# ============================================================

results = {}

for key, cfg in DATASETS.items():
    results[key] = process_dataset(
        dataset_key=key,
        rinex_path=cfg["rinex"],
        scenario_label=cfg["label"]
    )

combined_summary = pd.concat(
    [results[key]["summary"] for key in results],
    ignore_index=True
)

combined_summary_path = os.path.join(CSV_DIR, "combined_error_summary_open_vs_wall.csv")
combined_summary.to_csv(combined_summary_path, index=False)

print("Saved combined summary:")
print(combined_summary_path)

display(combined_summary)


In [ ]:
# ============================================================
# 5. Plot functions
# ============================================================

def save_fig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=250, bbox_inches="tight")
    print(f"Saved figure: {path}")
    plt.show()


def plot_baseline_enu(result, out_path, title):
    err = result["errors"]["OLS_L1"]

    plt.figure(figsize=(11, 5))
    plt.plot(err.index, err["E"], label="East error")
    plt.plot(err.index, err["N"], label="North error")
    plt.plot(err.index, err["U"], label="Up error")
    plt.xlabel("Epoch")
    plt.ylabel("Error [m]")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.xticks(rotation=45)
    save_fig(out_path)


def plot_wls_horizontal(result, out_path, title):
    e_ols = result["errors"]["OLS_L1"]
    e_wls = result["errors"]["WLS_soft_L1"]

    common_idx = e_ols.index.intersection(e_wls.index)
    e_ols = e_ols.loc[common_idx]
    e_wls = e_wls.loc[common_idx]

    plt.figure(figsize=(11, 5))
    plt.plot(e_ols.index, e_ols["horizontal_error"], label="OLS L1")
    plt.plot(e_wls.index, e_wls["horizontal_error"], label="WLS soft L1")
    plt.xlabel("Epoch")
    plt.ylabel("Horizontal error [m]")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.xticks(rotation=45)
    save_fig(out_path)


def plot_if_comparison(result, out_path, title):
    e_l1 = result["errors"]["OLS_L1"]
    e_if = result["errors"]["OLS_IF"]

    common_idx = e_l1.index.intersection(e_if.index)
    e_l1 = e_l1.loc[common_idx]
    e_if = e_if.loc[common_idx]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)

    axes[0].plot(e_l1.index, e_l1["horizontal_error"], label="OLS L1")
    axes[0].plot(e_if.index, e_if["horizontal_error"], label="OLS IF")
    axes[0].set_title("Horizontal error")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Error [m]")
    axes[0].grid(True)
    axes[0].legend()
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].plot(e_l1.index, e_l1["vertical_error"], label="OLS L1")
    axes[1].plot(e_if.index, e_if["vertical_error"], label="OLS IF")
    axes[1].set_title("Vertical error")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Error [m]")
    axes[1].grid(True)
    axes[1].legend()
    axes[1].tick_params(axis="x", rotation=45)

    fig.suptitle(title, y=1.04)
    save_fig(out_path)


def plot_hatch_if_comparison(result, out_path, title):
    e_if = result["errors"]["OLS_IF"]
    e_hatch = result["errors"]["HATCH_IF"]

    common_idx = e_if.index.intersection(e_hatch.index)
    e_if = e_if.loc[common_idx]
    e_hatch = e_hatch.loc[common_idx]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)

    axes[0].plot(e_if.index, e_if["horizontal_error"], label="OLS IF")
    axes[0].plot(e_hatch.index, e_hatch["horizontal_error"], label="HATCH IF")
    axes[0].set_title("Horizontal error")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Error [m]")
    axes[0].grid(True)
    axes[0].legend()
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].plot(e_if.index, e_if["vertical_error"], label="OLS IF")
    axes[1].plot(e_hatch.index, e_hatch["vertical_error"], label="HATCH IF")
    axes[1].set_title("Vertical error")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Error [m]")
    axes[1].grid(True)
    axes[1].legend()
    axes[1].tick_params(axis="x", rotation=45)

    fig.suptitle(title, y=1.04)
    save_fig(out_path)


def plot_summary_bar(combined_summary, out_path):
    methods = ["OLS_L1", "WLS_soft_L1", "OLS_IF", "HATCH_IF"]

    df = combined_summary[combined_summary["method"].isin(methods)].copy()
    df["method"] = pd.Categorical(df["method"], categories=methods, ordered=True)
    df = df.sort_values(["scenario", "method"])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    metrics = [
        ("rms_horizontal_error_m", "Horizontal RMS [m]"),
        ("rms_vertical_error_m", "Vertical RMS [m]"),
        ("rms_3d_error_m", "3D RMS [m]")
    ]

    scenarios = list(df["scenario"].unique())
    x = np.arange(len(methods))
    width = 0.35

    for ax, (metric, ylabel) in zip(axes, metrics):
        for i, scenario in enumerate(scenarios):
            sub = df[df["scenario"] == scenario].set_index("method").reindex(methods)
            ax.bar(x + (i - 0.5) * width, sub[metric].values, width, label=scenario)

        ax.set_xticks(x)
        ax.set_xticklabels(methods, rotation=30, ha="right")
        ax.set_ylabel(ylabel)
        ax.grid(True, axis="y")
        ax.legend()

    fig.suptitle("Open sky vs wall proximity: RMS error comparison", y=1.04)
    save_fig(out_path)


In [ ]:
# ============================================================
# 6. Save essential report plots
# ============================================================

# Open sky: keep these in the short report
plot_if_comparison(
    results["open_sky"],
    os.path.join(FIG_DIR, "open_sky_01_ionosphere_free_comparison.png"),
    "Open Sky: Effect of L1/L2 Ionosphere-Free Correction"
)

plot_hatch_if_comparison(
    results["open_sky"],
    os.path.join(FIG_DIR, "open_sky_02_hatch_if_comparison.png"),
    "Open Sky: Effect of Hatch Filtering on Ionosphere-Free Solution"
)

# Wall proximity: keep these in the short report
plot_baseline_enu(
    results["wall_proximity"],
    os.path.join(FIG_DIR, "wall_01_baseline_ols_l1_enu_errors.png"),
    "Wall Proximity: Baseline OLS L1 ENU Errors"
)

plot_hatch_if_comparison(
    results["wall_proximity"],
    os.path.join(FIG_DIR, "wall_02_hatch_if_comparison.png"),
    "Wall Proximity: Effect of Hatch Filtering on Ionosphere-Free Solution"
)

# Cross-scenario summary plot, optional but useful
plot_summary_bar(
    combined_summary,
    os.path.join(FIG_DIR, "open_vs_wall_rms_comparison.png")
)

# Backup plots, saved but not necessary for the short report
plot_wls_horizontal(
    results["open_sky"],
    os.path.join(FIG_DIR, "optional_open_sky_wls_horizontal.png"),
    "Open Sky: WLS Soft Horizontal Error Comparison"
)

plot_if_comparison(
    results["wall_proximity"],
    os.path.join(FIG_DIR, "optional_wall_ionosphere_free_comparison.png"),
    "Wall Proximity: Effect of L1/L2 Ionosphere-Free Correction"
)


In [ ]:
# ============================================================
# 7. Compact report table
# ============================================================

method_observations = {
    "OLS_L1": "Baseline standalone solution",
    "WLS_soft_L1": "Stable, but limited improvement",
    "OLS_IF": "Main accuracy improvement from ionospheric correction",
    "HATCH_IF": "Best final stability after Hatch smoothing"
}

report_table = combined_summary[
    combined_summary["method"].isin(["OLS_L1", "WLS_soft_L1", "OLS_IF", "HATCH_IF"])
].copy()

report_table["observation"] = report_table["method"].map(method_observations)

report_table = report_table[
    [
        "scenario",
        "method",
        "rms_horizontal_error_m",
        "rms_vertical_error_m",
        "rms_3d_error_m",
        "max_3d_error_m",
        "observation"
    ]
]

report_table = report_table.round({
    "rms_horizontal_error_m": 3,
    "rms_vertical_error_m": 3,
    "rms_3d_error_m": 3,
    "max_3d_error_m": 3
})

report_table_path = os.path.join(CSV_DIR, "report_error_comparison_table.csv")
report_table.to_csv(report_table_path, index=False)

print("Saved report table:")
print(report_table_path)

display(report_table)


In [ ]:
# ============================================================
# 8. LaTeX table template
# ============================================================

latex_table_template = r"""
\begin{table}[H]
\centering
\caption{Error comparison for open sky and wall proximity scenarios}
\renewcommand{\arraystretch}{1.2}
\begin{tabular}{p{2.5cm} p{2.7cm} p{2.3cm} p{2.3cm} p{2.3cm} p{4.0cm}}
\hline
\textbf{Scenario} & \textbf{Method} & \textbf{Horizontal RMS [m]} & \textbf{Vertical RMS [m]} & \textbf{3D RMS [m]} & \textbf{Main observation} \\
\hline
Open sky & OLS L1 & -- & -- & -- & Baseline standalone solution \\
Open sky & WLS soft L1 & -- & -- & -- & Stable, but limited improvement \\
Open sky & OLS IF & -- & -- & -- & Main accuracy improvement \\
Open sky & HATCH IF & -- & -- & -- & Best final stability \\
\hline
Wall proximity & OLS L1 & -- & -- & -- & Large wall-related error \\
Wall proximity & WLS soft L1 & -- & -- & -- & Slight improvement only \\
Wall proximity & OLS IF & -- & -- & -- & Reduces bias but remains noisy \\
Wall proximity & HATCH IF & -- & -- & -- & Smoother final result, but wall effects remain \\
\hline
\end{tabular}
\label{tab:error_comparison_open_wall}
\end{table}
"""

print(latex_table_template)


## Recommended figures for the short report

Use these four:

1. `open_sky_01_ionosphere_free_comparison.png`
2. `open_sky_02_hatch_if_comparison.png`
3. `wall_01_baseline_ols_l1_enu_errors.png`
4. `wall_02_hatch_if_comparison.png`

Use this table:

- `report_error_comparison_table.csv`

Optional backup figures are also saved, but they do not need to go into the short report.
